# پیوست الف: دیکشنری توابع کاربردی

هر سلول، نمونه‌ی کوتاه یک تابع است و برای اجرا به فایل ورودی خودتان نیاز دارد. جدول ب-۳ کتاب می‌گوید کدام نام‌ها نمادین‌اند.


## الف-۲) سامانه‌های مختصات و بازتصویر


In [ ]:
gdf_utm = gdf.to_crs("EPSG:32639")     # Tehran / Karaj zone
buffer = gdf_utm.geometry.buffer(500)  # 500 METRES  (correct)

# on EPSG:4326 the very same call would mean 500 DEGREES,
# which is a gross error

## الف-۴) شاخص‌های طیفی


In [ ]:
def add_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("NDWI")
    ndbi = img.normalizedDifference(["B11", "B8"]).rename("NDBI")
    return img.addBands([ndvi, ndwi, ndbi])
ic_with_indices = ic.map(add_indices)

## الف-۶) پیوند مکانی و آماره‌ی ناحیه‌ای


In [ ]:
from rasterstats import zonal_stats
import geopandas as gpd
stats = zonal_stats("counties.shp", "ndvi.tif", stats=["mean", "std"])
gdf = gpd.read_file("counties.shp")
gdf["ndvi_mean"] = [s["mean"] for s in stats]
gdf.to_file("counties_with_ndvi.shp")

## الف-۷) تقسیم Train/Test برای داده‌ی مکانی


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score

# build the groups: every sample gets the id of the 5 km tile it falls in
TILE_M = 5000                                  # tile edge, in metres
gdf_m = gdf.to_crs(epsg=32639)                 # a METRIC crs, never 4326
tile_x = (gdf_m.geometry.x // TILE_M).astype(int)
tile_y = (gdf_m.geometry.y // TILE_M).astype(int)
groups = tile_x.astype(str) + "_" + tile_y.astype(str)

gkf = GroupKFold(n_splits=5)
scores = []
for tr, te in gkf.split(X, y, groups=groups):
    m = RandomForestClassifier(n_estimators=200, random_state=42)
    m.fit(X.iloc[tr], y[tr])
    scores.append(f1_score(y[te], m.predict(X.iloc[te]), average="weighted"))
print(f"F1 = {np.mean(scores):.3f} +/- {np.std(scores):.3f}")

## الف-۸) یادگیری ماشین کلاسیک


In [ ]:
model = SomeClassifier(hyperparam=...)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

## الف-۹) کلان‌داده‌ی مکانی: وقتی RAM کافی نیست


In [ ]:
import rioxarray as rxr

# lazy: nothing is loaded into RAM yet
img = rxr.open_rasterio("s2_iran_50gb.tif",
                        chunks={"x": 1024, "y": 1024})

ndvi = ((img.sel(band=8) - img.sel(band=4))
        / (img.sel(band=8) + img.sel(band=4)))

ndvi.to_zarr("ndvi_iran.zarr")   # streamed to disk, never fully in RAM

## الف-۱۰) هوش مصنوعی تفسیرپذیر (XAI)


In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# which features drive the model overall:
shap.summary_plot(shap_values, X_test, feature_names=[
    "slope",
    "dist_river",
    "NDVI",
    "NDBI",
    "TWI",
])

## الف-۱۲) Embedding مکانی و یادگیری بازنمایی


In [ ]:
from geotessera.store import GeoTesseraZarr

gt = GeoTesseraZarr()

# Tehran, Karaj, Shiraz
points = [(51.39, 35.69), (50.99, 35.83), (52.58, 29.59)]
X = gt.sample_points(points, year=2025)   # shape: (3, 128)

from sklearn.cluster import KMeans
clusters = KMeans(n_clusters=5).fit_predict(X)   # feed into any model

## الف-۱۳) استقرار: ساخت یک Demo وب


In [ ]:
import streamlit as st
import geopandas as gpd
import folium
from streamlit_folium import st_folium

st.title("Urban Flood Risk Monitor")
district = st.sidebar.selectbox("district",
                                ["all", "Fardis", "Mehrshahr"])

gdf = gpd.read_file("flood_risk.geojson")
if district != "all":
    gdf = gdf[gdf["district"] == district]

st.metric("buildings at high risk", len(gdf[gdf["risk"] > 0.7]))

m = folium.Map(location=[35.83, 50.99], zoom_start=11)
folium.GeoJson(
    gdf,
    style_function=lambda x: {
        "fillColor": "red" if x["properties"]["risk"] > 0.7 else "yellow"
    },
).add_to(m)
st_folium(m, width=700)

# run with:  streamlit run app.py